# Phase 2: Contextual Embeddings with BERT

**Goal:** See how BERT generates *different* vectors for the same word in different contexts.

**What you'll learn:**
- How to load and use BERT from HuggingFace
- How tokenization works (including special [CLS] and [SEP] tokens)
- How to extract embeddings from BERT's hidden states
- Why 'bank' in 'river bank' has a different vector than 'bank' in 'bank account'
- The difference between [CLS] pooling and mean pooling

## Step 1: Install & Import

In [5]:
# Install (run once)
# !pip install transformers torch

In [6]:
import torch
import numpy as np
import matplotlib.pyplot as plt
from transformers import BertTokenizer, BertModel
from scipy.spatial.distance import cosine

print(f'PyTorch version: {torch.__version__}')

# Auto-detect best available device: CUDA (NVIDIA) → MPS (Apple Silicon) → CPU
if torch.cuda.is_available():
    device = 'cuda'
    print(f'Using device: CUDA (NVIDIA GPU) — {torch.cuda.get_device_name(0)}')
elif torch.backends.mps.is_available():
    device = 'mps'
    print(f'Using device: MPS (Apple Silicon GPU)')
else:
    device = 'cpu'
    print(f'Using device: CPU')

PyTorch version: 2.11.0
Using device: MPS (Apple Silicon GPU)


## Step 2: Load BERT Tokenizer and Model

We use `bert-base-uncased` — 12 transformer layers, 768 hidden dimensions, 110M parameters.
This will download ~440MB on first run (cached after that).

In [ ]:
model_name = 'bert-base-uncased'
print(f'Loading tokenizer: {model_name}...')
tokenizer = BertTokenizer.from_pretrained(model_name)

print(f'Loading model...')
model = BertModel.from_pretrained(model_name)
model.eval()  # Set to eval mode (disables dropout)
model = model.to(device)

print(f'Model loaded! Parameters: {sum(p.numel() for p in model.parameters()):,}')

Loading tokenizer: bert-base-uncased...


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Loading model...


config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

## Step 3: Understand Tokenization

BERT doesn't work with raw words — it uses a **WordPiece tokenizer** that splits unknown or rare words into subword pieces. Let's see how it works.

In [ ]:
sentences = [
    "I sat by the river bank",
    "I deposited money at the bank",
]

for sent in sentences:
    tokens = tokenizer.tokenize(sent)
    token_ids = tokenizer.encode(sent)
    decoded = tokenizer.convert_ids_to_tokens(token_ids)
    
    print(f'Sentence: {sent}')
    print(f'  Tokens:    {tokens}')
    print(f'  With [CLS]/[SEP]: {decoded}')
    print(f'  Token IDs: {token_ids}')
    print()

print('Note:')
print('  [CLS] = Classification token — added at the START of every input')
print('  [SEP] = Separator token — added at the END of every input')
print('  ## prefix = subword continuation (e.g., ##ing is the "-ing" part of a word)')

## Step 4: Extract BERT Embeddings

We'll write a helper function that:
1. Tokenizes the sentence
2. Runs it through BERT
3. Returns the full last hidden state (one vector per token)

In [ ]:
def get_bert_output(sentence):
    """
    Returns:
      tokens: list of token strings
      last_hidden: tensor of shape [num_tokens, 768]
      cls_embedding: the [CLS] token vector (shape [768])
      mean_embedding: mean of all token vectors (shape [768])
    """
    # Tokenize
    inputs = tokenizer(sentence, return_tensors='pt').to(device)
    tokens = tokenizer.convert_ids_to_tokens(inputs['input_ids'][0])

    # Run through BERT (no gradient needed for inference)
    with torch.no_grad():
        outputs = model(**inputs)

    # last_hidden_state shape: [batch=1, seq_len, hidden=768]
    last_hidden = outputs.last_hidden_state[0]  # Remove batch dim -> [seq_len, 768]

    cls_embedding  = last_hidden[0]             # [CLS] is always the first token
    mean_embedding = last_hidden.mean(dim=0)    # Average all token vectors

    return tokens, last_hidden, cls_embedding, mean_embedding

# Test it
tokens, hidden, cls_emb, mean_emb = get_bert_output("Hello world")
print(f'Tokens: {tokens}')
print(f'Hidden state shape: {hidden.shape}  (tokens × hidden_dim)')
print(f'[CLS] embedding shape: {cls_emb.shape}')
print(f'Mean embedding shape: {mean_emb.shape}')

## Step 5: The Key Demo — Same Word, Different Vectors

This is the core difference between Word2Vec and BERT.
Let's extract the embedding for the word 'bank' in two very different contexts.

In [ ]:
def cosine_similarity(v1, v2):
    """Compute cosine similarity between two vectors."""
    v1 = v1.cpu().numpy()
    v2 = v2.cpu().numpy()
    return 1 - cosine(v1, v2)  # scipy cosine returns distance, so 1 - distance = similarity

def get_word_embedding(sentence, target_word):
    """Get the embedding for a specific word in a sentence."""
    tokens, hidden, _, _ = get_bert_output(sentence)
    
    # Find the index of the target word in tokens
    # (target might be split into subwords — we take the first subword piece)
    target_idx = None
    for i, token in enumerate(tokens):
        if token.lower() == target_word.lower():
            target_idx = i
            break
    
    if target_idx is None:
        print(f'Warning: "{target_word}" not found as a token. Tokens: {tokens}')
        return None
    
    return hidden[target_idx]


# The two sentences
sent_river  = "I sat by the river bank yesterday"
sent_finance = "I deposited all my money at the bank"

bank_river   = get_word_embedding(sent_river, 'bank')
bank_finance = get_word_embedding(sent_finance, 'bank')

sim = cosine_similarity(bank_river, bank_finance)

print('BERT Contextual Embedding Demo')
print('=' * 50)
print(f'Sentence A: "{sent_river}"')
print(f'Sentence B: "{sent_finance}"')
print()
print(f'Cosine similarity of "bank" between A and B: {sim:.4f}')
print()
print('Interpretation:')
if sim < 0.85:
    print(f'  Score {sim:.4f} < 0.85: BERT correctly produces DIFFERENT vectors for each sense!')
else:
    print(f'  Score {sim:.4f}: Vectors are similar (small model, consider using larger BERT variant)')

## Step 6: Compare Multiple Contexts Side by Side

In [ ]:
# Let's run a broader comparison: word 'right' has many meanings
right_contexts = [
    ("Turn right at the next intersection", "right"),          # direction
    ("You have the right to remain silent", "right"),          # legal right
    ("That answer is right", "right"),                         # correct
    ("The right wing of the building is closed", "right"),     # side/wing
]

print('Cosine Similarity Matrix for "right" in different contexts:')
print()

# Get all embeddings
embeddings = []
labels = []
for sent, word in right_contexts:
    emb = get_word_embedding(sent, word)
    if emb is not None:
        embeddings.append(emb)
        labels.append(sent[:40] + '...')

# Build similarity matrix
n = len(embeddings)
sim_matrix = np.zeros((n, n))
for i in range(n):
    for j in range(n):
        sim_matrix[i][j] = cosine_similarity(embeddings[i], embeddings[j])

# Display
fig, ax = plt.subplots(figsize=(10, 4))
im = ax.imshow(sim_matrix, cmap='YlOrRd', vmin=0.7, vmax=1.0)
ax.set_xticks(range(n))
ax.set_yticks(range(n))
ax.set_xticklabels([f'Ctx {i+1}' for i in range(n)], rotation=15)
ax.set_yticklabels([f'Ctx {i+1}: {l}' for i, l in enumerate(labels)], fontsize=9)
plt.colorbar(im, ax=ax)
ax.set_title('Similarity of "right" across different contexts\n(lower = more different meanings)', fontsize=12)

for i in range(n):
    for j in range(n):
        ax.text(j, i, f'{sim_matrix[i][j]:.2f}', ha='center', va='center', fontsize=10)

plt.tight_layout()
plt.savefig('phase2_context_similarity.png', dpi=150)
plt.show()

## Step 7: [CLS] vs Mean Pooling for Sentence Embeddings

In [ ]:
test_sentences = [
    "The cat sat on the mat",
    "A kitten rested on the rug",       # semantically similar to above
    "The stock market crashed today",   # unrelated
]

cls_embeddings  = []
mean_embeddings = []

for sent in test_sentences:
    _, _, cls_emb, mean_emb = get_bert_output(sent)
    cls_embeddings.append(cls_emb)
    mean_embeddings.append(mean_emb)

print('Sentence similarity comparison:')
print(f'  S1: "{test_sentences[0]}"')
print(f'  S2: "{test_sentences[1]}"  (should be SIMILAR to S1)')
print(f'  S3: "{test_sentences[2]}"  (should be DIFFERENT from S1)')
print()

for method, embeds in [('[CLS] pooling', cls_embeddings), ('Mean pooling', mean_embeddings)]:
    s1_s2 = cosine_similarity(embeds[0], embeds[1])
    s1_s3 = cosine_similarity(embeds[0], embeds[2])
    print(f'{method}:')
    print(f'  S1 ↔ S2 (similar): {s1_s2:.4f}')
    print(f'  S1 ↔ S3 (different): {s1_s3:.4f}')
    print()

print('Takeaway: Mean pooling often gives better sentence-level representations than [CLS].')
print('For best results, Phase 3 uses SBERT — which is BERT specifically fine-tuned for this!')

## Step 8: Visualize How Layers Build Up Understanding

BERT has 12 layers. Early layers capture syntax, later layers capture semantics.
Let's see how the representation of 'bank' changes across layers.

In [ ]:
def get_all_layer_embeddings(sentence, target_word):
    """Get the embedding for target_word at every BERT layer."""
    inputs = tokenizer(sentence, return_tensors='pt').to(device)
    tokens = tokenizer.convert_ids_to_tokens(inputs['input_ids'][0])

    # output_hidden_states=True returns all 13 hidden states (embedding + 12 layers)
    with torch.no_grad():
        outputs = model(**inputs, output_hidden_states=True)

    target_idx = next((i for i, t in enumerate(tokens) if t.lower() == target_word.lower()), None)
    if target_idx is None:
        return None, None

    # hidden_states is a tuple of 13 tensors, each [batch, seq, 768]
    layer_embeddings = [hs[0, target_idx].cpu().numpy() for hs in outputs.hidden_states]
    return layer_embeddings, tokens

# Get layer-by-layer embeddings for 'bank' in both contexts
layers_river,   _ = get_all_layer_embeddings(sent_river, 'bank')
layers_finance, _ = get_all_layer_embeddings(sent_finance, 'bank')

# Compute cosine similarity at each layer
layer_sims = []
for l_r, l_f in zip(layers_river, layers_finance):
    sim = 1 - cosine(l_r, l_f)
    layer_sims.append(sim)

# Plot
plt.figure(figsize=(10, 5))
plt.plot(range(13), layer_sims, 'o-', color='royalblue', linewidth=2, markersize=8)
plt.axhline(y=layer_sims[0], color='gray', linestyle='--', alpha=0.5, label='Layer 0 (input embedding)')
plt.xlabel('BERT Layer (0 = input embedding, 12 = final layer)', fontsize=12)
plt.ylabel('Cosine Similarity', fontsize=12)
plt.title('How "bank" diverges across BERT layers\n"river bank" vs "bank account"\n(lower = model has learned they are different)', fontsize=12)
plt.xticks(range(13))
plt.ylim(0.5, 1.05)
plt.grid(True, alpha=0.3)
plt.legend()
plt.tight_layout()
plt.savefig('phase2_layer_divergence.png', dpi=150)
plt.show()

print('As we go deeper in BERT, the two senses of "bank" diverge more.')
print('Early layers handle syntax; later layers encode meaning.')

## Key Takeaways

1. **BERT tokenizes with WordPiece** — every input gets [CLS] and [SEP] tokens, rare words are split into subwords
2. **Contextual embeddings** — the same word 'bank' gets different vectors depending on surrounding context
3. **12 layers build up meaning** — early layers = syntax, later layers = semantics
4. **Two pooling strategies** — [CLS] token or mean of all tokens; both work but mean pooling often edges out [CLS]
5. **BERT is a foundation, not a solution** — for *sentence-level* similarity tasks, fine-tuned SBERT (Phase 3) is far better

**→ Phase 3 will use SBERT to get high-quality sentence embeddings ready for RAG!**